In [ ]:
from FIT_python.pipeline_sex.data_import_utils import load_raw_files
from FIT_python.config import RAW_DIR
raw_dfs = load_raw_files(RAW_DIR)
raw_dfs

In [ ]:
# %% [code]
from pathlib import Path
import pandas as pd
import numpy as np
import ast
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterSampler

from FIT_python.pipeline_sex.pipeline_wrapper_sex import PipelineWrapper
from FIT_python.config import RESULTS_DATA_DIR

# 1) Prepare (Clean → Split → Summary)
prep = PipelineWrapper(
    model_keys=["log_reg"],  # Dummy
    impute_method=None,
    outlier_method=None,
    scaler_method=None,
    reduce_pre_method=None,
    reduce_post_method=None,
    fs_method=None,
    fs_k=None,
)
prep.prepare()



In [4]:
# %% [code] RandomizedSearchCV mit vordefiniertem Train/Test und Individual‐Balanced‐Sex‐Score
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from FIT_python.pipeline_sex.individual_scorers import individual_balanced_sex
from FIT_python.pipeline_sex.pipeline_wrapper_sex import get_pipeline_steps
from FIT_python.pipeline_sex.models import MODELS
from FIT_python.config import SPLITS_DIR, RESULTS_DATA_DIR

# 1) Modell‐Keys
model_keys = [
    "logreg_l2","logreg_l1",
    "rf_small","rf_med","rf_large",
    "et_med","et_sm",
    "knn_3","knn_5","knn_7",
    "svm_linear","svm_rbf",
    "xgb_std","xgb_hist",
    "lgbm_std","lgbm_md10",
    "catb_std","catb_fast","lda",
]

# 2) Parameterräume
param_distributions = {
    "clf":                [MODELS[k] for k in model_keys],
    "select__method":     [None,"forward","random_forest","variance","univariate","lasso"],
    "select__k":          [1,2,3,5,10,20,50],
    "outlier__method":    [None,"clip","zscore"],
    "scale__method":      [None,"standard","robust"],
    "reduce_pre__method": [None,"pca","umap"],
    "reduce_post__method":[None,"pca","umap"],
}

# 3) Ergebnis‐Ordner
out_dir = Path(RESULTS_DATA_DIR) / "random_search_individual_sex"
out_dir.mkdir(parents=True, exist_ok=True)

# 4) Loop über Spezies
for species_dir in sorted(Path(SPLITS_DIR).iterdir()):
    if not species_dir.is_dir():
        continue
    species = species_dir.name
    print(f"\n▶ RandomizedSearch für {species}")

    # 4.1) Train/Test laden
    df_train = pd.read_parquet(species_dir / "train.parquet").query("sex in ['f','m']")
    df_test  = pd.read_parquet(species_dir / "test.parquet").query("sex in ['f','m']")

    # 4.2) Features & Ziel & IDs
    X_tr = df_train.select_dtypes(include="number")
    y_tr = df_train["sex"].map({"f":0,"m":1}).values
    ids_tr = df_train["individual_id"].values

    X_te = df_test.select_dtypes(include="number")
    y_te = df_test["sex"].map({"f":0,"m":1}).values
    ids_te = df_test["individual_id"].values

    # 4.3) Basis‐Pipeline ohne Klassifikator
    steps = get_pipeline_steps(
        fs_method=None, fs_k=None,
        impute_method=None, outlier_method=None,
        scaler_method=None,
        reduce_pre_method=None, reduce_post_method=None
    )
    steps.append(('clf', MODELS['rf_small']))
    pipe = Pipeline(steps)
    # 4.4) RandomizedSearchCV aufsetzen
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=50,
        scoring=individual_balanced_sex,
        cv=5,
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    # 4.5) Fit auf vordefiniertem Train (mit individual_ids)
    search.fit(X_tr, y_tr, individual_ids=ids_tr)

    # 4.6) Best‐Ergebnisse speichern
    best = {
        "species":    species,
        "best_score": search.best_score_,
        "best_params": search.best_params_
    }
    pd.DataFrame([best]).to_csv(out_dir / f"{species}_best_individual_sex.csv", index=False)

    # 4.7) Final‐Score auf Test
    y_te_pred = search.best_estimator_.predict(X_te)
    ibs_test = individual_balanced_sex._score_func(y_te, y_te_pred, ids_te)
    print(f" ✓ {species}: IBS_train={search.best_score_:.4f}, IBS_test={ibs_test:.4f}")



▶ RandomizedSearch für aj_someringii
Fitting 5 folds for each of 50 candidates, totalling 250 fits


ValueError: Invalid parameter 'select' for estimator Pipeline(steps=[('transform', NumericTransformer()),
                ('clf',
                 <catboost.core.CatBoostClassifier object at 0x000001AE3C01B650>)]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].

In [ ]:
import pandas as pd
from joblib import load

# Pfade definieren
model_path = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/sex_models_best/eurasian_otter.joblib"
data_path  = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/raw/Eurasian Otter.csv"
output_csv = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/eurasian_otter_predictions.csv"

# Modell laden
clf = load(model_path)

# Daten laden und Spaltennamen anpassen (Punkte, Bindestriche → Unterstriche; T → t)
df = pd.read_csv(data_path)
df.columns = [
    col.replace('.', '_')
       .replace('-', '_')
       .replace('T', 't')
    for col in df.columns
]

# Direkt Vorhersagen aus der Pipeline holen
df["predicted_sex"]     = clf.predict(df)
probas = clf.predict_proba(df)
df["predicted_proba_f"] = probas[:, 0]
df["predicted_proba_m"] = probas[:, 1]

# Ergebnisse als CSV speichern
df.to_csv(output_csv, index=False)
print(f"Ergebnisse gespeichert in: {output_csv}")
